# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ankita0531/ML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Unit of analysis:** One analytical row represents one content item for one client (`client_hash_id × content_hash_id`). The source table is daily, so daily records are aggregated into separate feature and label windows.

**Feature window:** February 1, 2026 to February 28, 2026.

**Label window:** March 1, 2026 to March 31, 2026.

March 2026 is the mid-panel development month used to iterate and verify the label logic. The June 2026 `_sample` is treated as a sealed test month and is not used for feature or label development.

In [22]:
%pip -q install duckdb

In [23]:
import duckdb

con = duckdb.connect()

print("DuckDB connected.")

DuckDB connected.


In [24]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF token loaded:", HF_TOKEN is not None)

HF token loaded: True


In [25]:
con.execute(f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
""")

print("Hugging Face authentication configured.")

Hugging Face authentication configured.


In [26]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
REL = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"

check = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM {REL}
""").df()

check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31


In [27]:
grain_check = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS n
    FROM {REL}
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,n


In [28]:
columns = con.sql(f"""
    DESCRIBE SELECT *
    FROM {REL}
""").df()

columns[["column_name", "column_type"]]

,column_name,column_type
0,report_date,DATE
1,client_hash_id,VARCHAR
2,content_hash_id,VARCHAR
3,client_has_gsc,BOOLEAN
4,client_has_ga4,BOOLEAN
5,gsc_data_available,BOOLEAN
6,ga4_data_available,BOOLEAN
7,gsc_impressions,BIGINT
8,gsc_clicks,BIGINT
9,gsc_sum_position,BIGINT


## 2. Fields: feature / label / context / excluded

**Features:** `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, `ga4_sessions`, `ga4_engaged_sessions`. These are aggregated from the February 2026 feature window and are knowable at the decision moment because they describe performance observed before the March prediction window.

**Label/proxy:** A content item is labeled as declining when March 2026 GSC impressions are more than 20% lower than February 2026 GSC impressions.

**Context:** `client_hash_id`, `content_hash_id`, and the feature/label window dates. These identify the content-client pair and the time periods used for the analysis.

**Excluded:** Future March 2026 performance fields used to construct the label are excluded from the feature set because they would leak the outcome into the prediction.

In [29]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
feature_label = con.sql(f"""
WITH feb AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS gsc_impressions,
        SUM(gsc_clicks) AS gsc_clicks,
        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN SUM(gsc_sum_position) * 1.0 / SUM(gsc_impressions)
            ELSE NULL
        END AS gsc_avg_position,
        SUM(ga4_sessions) AS ga4_sessions,
        SUM(ga4_engaged_sessions) AS ga4_engaged_sessions
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet'
    )
    WHERE gsc_data_available IS TRUE
      AND ga4_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
),

mar AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS march_gsc_impressions
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
)

SELECT
    f.client_hash_id,
    f.content_hash_id,
    f.gsc_impressions,
    f.gsc_clicks,
    f.gsc_avg_position,
    f.ga4_sessions,
    f.ga4_engaged_sessions,
    CASE
        WHEN m.march_gsc_impressions < 0.8 * f.gsc_impressions
        THEN 1
        ELSE 0
    END AS is_declining_proxy
FROM feb f
INNER JOIN mar m
    ON f.client_hash_id = m.client_hash_id
   AND f.content_hash_id = m.content_hash_id
WHERE f.gsc_impressions > 0
""").df()

feature_label.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions,is_declining_proxy
0,client_9958f0a7ae1df715,content_810cf06597918291,192.0,0.0,7.880208,22.0,8.0,0
1,client_9958f0a7ae1df715,content_1d69c2ed06358f6f,603.0,0.0,7.671642,17.0,4.0,1
2,client_9958f0a7ae1df715,content_55ead56c1217a888,75.0,3.0,8.693333,8.0,0.0,0
3,client_9958f0a7ae1df715,content_ea2ccd069e9caaf7,16.0,0.0,5.812500,2.0,0.0,0
4,client_9958f0a7ae1df715,content_b813c73d7000b3b1,274.0,4.0,8.000000,12.0,1.0,0


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [30]:
grain_check = con.sql("""
    SELECT
        client_hash_id,
        content_hash_id,
        COUNT(*) AS n
    FROM feature_label
    GROUP BY client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

grain_check

,client_hash_id,content_hash_id,n


In [31]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
window_check = con.sql("""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
""").df()

window_check

,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31


In [32]:
availability_check = con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
              AND ga4_data_available IS TRUE
        ) AS both_available_rows
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
""").df()

availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows,both_available_rows
0,9841378,3611061,364347


## 4. Data limits

**Limitation:** Data availability is uneven across clients. In March 2026, only 3,611,061 of 9,841,378 rows have GSC data available, and only 364,347 have both GSC and GA4 available. Therefore, the feature frame represents only client-content pairs with usable data in the required windows.

The June 2026 `_sample` is treated as a sealed test month and is not used while developing the contract or label logic.

In [33]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
leakage_test = feature_label.copy()

# Deliberate leakage: use the target itself as a feature
leakage_test["leaky_feature"] = leakage_test["is_declining_proxy"]

leakage_test.head()

,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions,is_declining_proxy,leaky_feature
0,client_9958f0a7ae1df715,content_810cf06597918291,192.0,0.0,7.880208,22.0,8.0,0,0
1,client_9958f0a7ae1df715,content_1d69c2ed06358f6f,603.0,0.0,7.671642,17.0,4.0,1,1
2,client_9958f0a7ae1df715,content_55ead56c1217a888,75.0,3.0,8.693333,8.0,0.0,0,0
3,client_9958f0a7ae1df715,content_ea2ccd069e9caaf7,16.0,0.0,5.812500,2.0,0.0,0,0
4,client_9958f0a7ae1df715,content_b813c73d7000b3b1,274.0,4.0,8.000000,12.0,1.0,0,0


In [34]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

X = leakage_test[[
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions",
    "leaky_feature"
]]

y = leakage_test["is_declining_proxy"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = DecisionTreeClassifier(max_depth=3, random_state=42)
model.fit(X_train, y_train)

pred = model.predict(X_test)

leaky_accuracy = accuracy_score(y_test, pred)

print("Accuracy with leakage:", round(leaky_accuracy, 4))

Accuracy with leakage: 1.0


In [35]:
honest_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions"
]

X_honest = feature_label[honest_features]
y_honest = feature_label["is_declining_proxy"]

X_train, X_test, y_train, y_test = train_test_split(
    X_honest, y_honest,
    test_size=0.2,
    random_state=42,
    stratify=y_honest
)

honest_model = DecisionTreeClassifier(max_depth=3, random_state=42)
honest_model.fit(X_train, y_train)

honest_pred = honest_model.predict(X_test)

honest_accuracy = accuracy_score(y_test, honest_pred)

print("Accuracy without leakage:", round(honest_accuracy, 4))

Accuracy without leakage: 0.9819


## Leakage experiment

A deliberate leakage feature, `leaky_feature`, was created directly from the label `is_declining_proxy`. A decision-tree model achieved **1.0000 accuracy** when this feature was included, demonstrating that future/label-derived information can make model performance look artificially perfect.

After removing the leakage feature and retaining only the five pre-decision features, the same model achieved **0.9817 accuracy**. The honest score is therefore **0.9817**, while the 1.0000 score is rejected because it depends on target leakage.

## Feature timing: knowable at the decision moment

- `gsc_impressions` — Knowable at the decision moment because it is aggregated from GSC observations in the February 2026 feature window, before the March prediction window.
- `gsc_clicks` — Knowable at the decision moment because it is aggregated from GSC observations available before the March prediction window.
- `gsc_avg_position` — Knowable at the decision moment because it is calculated from February GSC observations using information available before prediction.
- `ga4_sessions` — Knowable at the decision moment because it is aggregated from GA4 observations in the February feature window.
- `ga4_engaged_sessions` — Knowable at the decision moment because it is aggregated from GA4 observations in the February feature window.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.